In [5]:
import transformers

## Import Dataset

In [2]:
from google.colab import files
uploaded = files.upload()

Saving chat_data.jsonl to chat_data.jsonl


In [6]:
from datasets import load_dataset

In [7]:

dataset = load_dataset("HuggingFaceH4/ultrachat_200k")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


In [8]:

from pprint import pprint

pprint(dataset["train_sft"][0])

{'messages': [{'content': 'These instructions apply to section-based themes '
                          '(Responsive 6.0+, Retina 4.0+, Parallax 3.0+ Turbo '
                          '2.0+, Mobilia 5.0+). What theme version am I '
                          'using?\n'
                          'On your Collections pages & Featured Collections '
                          'sections, you can easily show the secondary image '
                          'of a product on hover by enabling one of the '
                          "theme's built-in settings!\n"
                          'Your Collection pages & Featured Collections '
                          'sections will now display the secondary product '
                          'image just by hovering over that product image '
                          'thumbnail.\n'
                          'Does this feature apply to all sections of the '
                          'theme or just specific ones as listed in the text '
                    

In [9]:
from pprint import pprint
pprint(dataset['train_sft']['messages'])

Column([[{'content': "These instructions apply to section-based themes (Responsive 6.0+, Retina 4.0+, Parallax 3.0+ Turbo 2.0+, Mobilia 5.0+). What theme version am I using?\nOn your Collections pages & Featured Collections sections, you can easily show the secondary image of a product on hover by enabling one of the theme's built-in settings!\nYour Collection pages & Featured Collections sections will now display the secondary product image just by hovering over that product image thumbnail.\nDoes this feature apply to all sections of the theme or just specific ones as listed in the text material?", 'role': 'user'}, {'content': 'This feature only applies to Collection pages and Featured Collections sections of the section-based themes listed in the text material.', 'role': 'assistant'}, {'content': 'Can you guide me through the process of enabling the secondary image hover feature on my Collection pages and Featured Collections sections?', 'role': 'user'}, {'content': "Sure, here are 

In [10]:
data = []
for conversation in dataset['train_sft']:
    # Stop if we already have 1000 pairs
    if len(data) >= 1000:
        break

    msgs = conversation['messages']
    for i in range(len(msgs) - 1):
        if msgs[i]['role'] == 'user' and msgs[i+1]['role'] == 'assistant':
            data.append({
                'input': f"User: {msgs[i]['content']}\nBot:",
                'output': msgs[i+1]['content']
            })
            # Check again inside the loop to be precise
            if len(data) >= 1000:
                break

print(f"Total examples in data: {len(data)}")

Total examples in data: 1000


In [11]:
# Next steps -> convert normal data into input output format -> import datasets Dataset and convert into proper dataset!!

In [12]:
from transformers import AutoTokenizer , AutoModelForSeq2SeqLM

In [13]:
model_name = "google/flan-t5-small"

In [14]:
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSeq2SeqLM.from_pretrained(model_name)

Loading weights:   0%|          | 0/190 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


In [15]:
from transformers import DataCollatorForSeq2Seq

data_collator = DataCollatorForSeq2Seq(
    tokenizer=tokenizer,
    model=model
)

In [16]:
def tokenize(data):
    # Tokenize the input
    model_inputs = tokenizer(
        data['input'],
        truncation=True,
        max_length=200
    )

    # Tokenize the target/labels
    labels = tokenizer(
        data['output'],
        truncation=True,
        max_length=200
    )

    # Replace padding token ids with -100 so they are ignored by the loss function

    model_inputs['labels'] = labels['input_ids']
    return model_inputs



In [20]:
from datasets import Dataset
dataset = Dataset.from_list(data)
tokenized_data = dataset.map(tokenize, batched=False , remove_columns = dataset.column_names)
tokenized_data.set_format("torch")

Map:   0%|          | 0/1000 [00:00<?, ? examples/s]

In [21]:
tokenized_data[0]

{'input_ids': tensor([ 6674,    10,   506,  3909,  1581,    12,  1375,    18,   390,  8334,
            41,  1649,     7,  5041,     7,   757,     3, 22642,  1220,     6,
           419,    17,    77,     9,     3, 15021,  1220,     6,  4734,   195,
             9,   226,  1877,   632,  1220, 22299,  6864,  1220,     6,  1290,
           115, 13565,     3, 20734,  1220,   137,   363,  3800,   988,   183,
            27,   338,    58,   461,    39,  6767,     7,  1688,     3,   184,
             3, 16772,    26,  6767,     7,  6795,     6,    25,    54,  1153,
           504,     8,  6980,  1023,    13,     3,     9,   556,    30, 21994,
            57,     3, 10165,    80,    13,     8,  3800,    31,     7,  1192,
            18,    77,  3803,    55,   696,  6767,  1688,     3,   184,     3,
         16772,    26,  6767,     7,  6795,    56,   230,  1831,     8,  6980,
           556,  1023,   131,    57, 21994,    53,   147,    24,   556,  1023,
         30250,     5,  3520,    48,  1

In [22]:
from transformers import TrainingArguments

In [23]:
training_args = TrainingArguments(
    output_dir="./results",
    learning_rate=3e-4,
    per_device_train_batch_size=4,
    num_train_epochs=3,
    logging_steps=1,
    save_strategy="epoch",
    report_to="none"
)

In [25]:
from transformers import Trainer

In [26]:
from transformers import DataCollatorForSeq2Seq

# 1. Initialize the collator (it needs the tokenizer to know the pad_token_id)
data_collator = DataCollatorForSeq2Seq(tokenizer, model=model)

# 2. Pass it into your Trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_data,  # Your list of tokenized dicts
    data_collator=data_collator    # <--- This is the missing piece!
)

trainer.train()

/usr/local/lib/python3.12/dist-packages/transformers/data/data_collator.py:600: UserWarning: Creating a tensor from a list of numpy.ndarrays is extremely slow. Please consider converting the list to a single numpy.ndarray with numpy.array() before converting to a tensor. (Triggered internally at /pytorch/torch/csrc/utils/tensor_new.cpp:253.)
  batch["labels"] = torch.tensor(batch["labels"], dtype=torch.int64)


Step,Training Loss
1,9.694890
2,9.674180
3,9.650416
4,9.621324
5,9.496437
6,9.489838
7,9.355575
8,9.311506
9,9.302880
10,9.285557


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=750, training_loss=6.690617288589477, metrics={'train_runtime': 159.1025, 'train_samples_per_second': 18.856, 'train_steps_per_second': 4.714, 'total_flos': 126741017468928.0, 'train_loss': 6.690617288589477, 'epoch': 3.0})

In [32]:

# these are the additional tokens which your input text will be added with, change accordingly
extra = ["User:", "\nBot:"]

def test_model(user_input: str):
    # Create the combined string using a distinct variable name
    full_text = user_input #extra[0] + " " + user_input + extra[-1]

    # Tokenize and place onto GPU
    inputs = tokenizer(full_text, return_tensors="pt").to('cuda')

    # Generate response (pass the dictionary using **inputs)
    output = model.generate(
    **inputs,
    max_length=200,
    do_sample=True,   # Makes it creative
    temperature=0.7, # Lower = more focused, Higher = more chaotic
    top_k=50
)

    # Decode and print
    print(tokenizer.decode(output[0], skip_special_tokens=True))

test_model("I am feeling lonely...")

as to as we don'll be of the best.


In [33]:

test_model("How do you deal with anxiety especially when you are under a lot of pressure.")


What as and when more needs in time on your hand the. The that is the time you is an ideal type of pressure. "Welkdenkp'dref eunde,' the not it,' as san-end., is it the time for int for this, and you has a that would find dmts in your heart.


In [40]:
import random
id = random.randint(0,1000)
dataset[id]['input']

"User: Ugh, I don't want to go to any of those places. Can you recommend something more exciting to do in Neuchâtel?\nBot:"

In [41]:
test_model(dataset[id]['input'])

I should include with and a cd. For a great family, a tolss to be mwa a fvg's Isiie, or lsa to get in the heart of "Fiein", a family. There were to be a inial the news it is also the real place to add to a place to learn a time for a anewesss. This or-ma smalha tsa to is bsl, a yo-bing on a a mina. To help use


In [38]:
test_model("How are you?")

The A. s if also an x, a asy 're and a x of bes' was.


In [39]:
test_model( "I'm craving something sweet right now.")